# 📊 Notebook 01 — Generación de Datos Sintéticos para Panam

**Proyecto:** Data Seekers — Revolución NoSQL para Panam  
**Objetivo:** Simular las cuatro fuentes de datos del ecosistema Panam con realismo suficiente para que el pipeline ETL posterior tenga que hacer **minería de texto real** y aprovechar las propiedades de las bases NoSQL.

---

## 🎯 ¿Qué se genera aquí?

| Dataset | Registros | Destino | Justificación NoSQL |
|---|---|---|---|
| **Reseñas** | 1,000 | MongoDB | Texto semiestructurado (emojis, hashtags, CAPS, longitud variable) |
| **Ventas** | 5,000 | Cassandra | Series temporales por sucursal/modelo |
| **Inventario** | 2,000 | Cassandra | Movimientos de stock por estado |
| **Eventos Web** | 800 | MongoDB | Eventos heterogéneos del comportamiento de usuarios |

## 🧠 Decisión de diseño clave: las reseñas NO traen sentimiento etiquetado

En la versión anterior cada reseña tenía un campo `sentimiento` asignado al azar. Eso **rompe la narrativa NoSQL**: si los datos llegan ya clasificados, no hay valor en hacer minería de texto y daría igual usar una base relacional.

En esta versión:
- Las reseñas son texto crudo de redes sociales (con emojis, hashtags, CAPS, mezcla de buenos y malos puntos).
- El **ETL del notebook 02** es quien aplica análisis léxico + scoring de emojis + detección de amplificadores para derivar el sentimiento.
- Conservamos un campo oculto `intencion_plantilla` como *ground truth* opcional para validar la precisión del clasificador y mostrarlo en el dashboard.

## 📅 Rango temporal

Todas las fechas cubren los **últimos 5 años** para permitir análisis estacionales, tendencias y comparativas año contra año.


## 1️⃣ Imports y configuración inicial

> 💡 Si `faker` no está instalado, descomenta la primera línea.

In [1]:
# !pip install faker

import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta
import random

# Semilla para reproducibilidad
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

fake = Faker('es_MX')
Faker.seed(SEED)

print("✅ Librerías cargadas correctamente")

✅ Librerías cargadas correctamente


## 2️⃣ Resolución de la raíz del proyecto

Usamos `get_project_root()` para que el notebook funcione sin importar desde qué subcarpeta se ejecute. Busca el `README.md` raíz hacia arriba.

In [2]:
from pathlib import Path

def get_project_root(marker: str = "README.md") -> Path:
    """Sube por el árbol de directorios hasta encontrar el marcador de raíz."""
    current = Path.cwd()
    for parent in [current] + list(current.parents):
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError("No se encontró la raíz del proyecto.")

PROJECT_ROOT = get_project_root()
DATOS = PROJECT_ROOT / "data"
DATOS.mkdir(exist_ok=True)

print(f"📂 Raíz del proyecto: {PROJECT_ROOT}")
print(f"📂 Carpeta de datos:  {DATOS}")

📂 Raíz del proyecto: c:\Users\PC\Desktop\Antigravity\Panam_BDN
📂 Carpeta de datos:  c:\Users\PC\Desktop\Antigravity\Panam_BDN\data


## 3️⃣ Catálogo real de Panam

Estos son los modelos, sucursales y colores reales de Panam. Los **estados** se derivan automáticamente del prefijo de cada sucursal — así evitamos duplicar la fuente de verdad y los almacenes del inventario quedan automáticamente alineados con los puntos de venta.

In [3]:
# 👟 Modelos reales de Panam (16 líneas)
MODELS = [
    '084', '084 Campeón', '084 Clásico', '084 Diamante', 'Panam Alfa',
    'Vaivén', 'Urbano', 'Vital', 'Ultra', 'Barracuda',
    'Flip', 'Gran Prix', 'RL15', 'Caimán', 'Perestroika', 'SuperFaro'
]

# 🎨 Colores disponibles
COLORS = ['negro', 'blanco', 'rojo', 'azul', 'gris', 'verde', 'rosa', 'beige']

# 📍 Sucursales reales de Panam (49 puntos de venta)
SUCURSALES = [
    'CDMX_20_de_Noviembre', 'CDMX_5_de_Mayo', 'CDMX_Centro', 'CDMX_Condesa',
    'CDMX_Del_Valle', 'CDMX_Insurgentes', 'CDMX_La_Villa', 'CDMX_Mixcoac',
    'CDMX_Polanco', 'CDMX_Regina', 'CDMX_San_Cosme', 'CDMX_San_Jacinto',
    'CDMX_Universidad', 'CDMX_Zona_Rosa', 'CDMX_Madero', 'CDMX_Rep_del_Salvador',
    'CDMX_Pino_Suarez', 'CDMX_Huipulco', 'CDMX_Tacubaya', 'CDMX_Metrobus_Chilpancingo',
    'CDMX_Metrobus_Durango', 'CDMX_Plaza_Ermita', 'CDMX_Satelite',
    'EdoMex_Americas_Ecatepec', 'EdoMex_Atlacomulco', 'EdoMex_Chalco',
    'EdoMex_Chimalhuacan', 'EdoMex_Nezahualcoyotl', 'EdoMex_Nicolas_Romero',
    'EdoMex_Plaza_Aragon', 'EdoMex_Sendero_Ixtapaluca', 'EdoMex_Sendero_Toluca',
    'EdoMex_Tenayuca', 'EdoMex_Tepeyac', 'EdoMex_Texcoco',
    'EdoMex_Toluca', 'EdoMex_Toluca_Grand_Plaza', 'EdoMex_Zumpango',
    'EdoMex_Patio_Valle_De_Chalco',
    'Chihuahua_Sendero', 'Hidalgo_Pachuca', 'Hidalgo_Tula',
    'Michoacan_Morelia_Centro', 'Michoacan_Uruapan', 'Morelos_Cuernavaca',
    'Nuevo_Leon_Monterrey', 'Puebla_Centro', 'Puebla_Tehuacan',
    'Quintana_Roo_Playa_del_Carmen', 'Coahuila_Saltillo'
]

# 🗺️ Estados derivados (única fuente de verdad: el prefijo de la sucursal)
ESTADOS = sorted({s.split('_')[0] for s in SUCURSALES})

# 📱 Plataformas donde aparecen las reseñas
FUENTES = ['instagram', 'tiktok', 'twitter', 'facebook', 'google_reviews', 'amazon']

print(f"👟 Modelos:    {len(MODELS)}")
print(f"📍 Sucursales: {len(SUCURSALES)}")
print(f"🗺️  Estados:    {len(ESTADOS)} -> {ESTADOS}")
print(f"📱 Fuentes:    {FUENTES}")

👟 Modelos:    16
📍 Sucursales: 50
🗺️  Estados:    10 -> ['CDMX', 'Chihuahua', 'Coahuila', 'EdoMex', 'Hidalgo', 'Michoacan', 'Morelos', 'Nuevo', 'Puebla', 'Quintana']
📱 Fuentes:    ['instagram', 'tiktok', 'twitter', 'facebook', 'google_reviews', 'amazon']


## 4️⃣ Plantillas de reseñas realistas

Aquí está el corazón del ejercicio NoSQL: **45 plantillas** organizadas por intención original. Cada una incluye señales que el ETL tendrá que detectar:

- **Emojis con valencia** (😍 ❤️ 🔥 vs 😡 👎 💔)
- **CAPS LOCK** como amplificador de emoción
- **Múltiples signos de exclamación** (`!!!`)
- **Hashtags** (`#Panam`, `#malacompra`)
- **Negaciones** (`no compren`, `nunca más`)
- **Reseñas mixtas** que combinan elogios y quejas (caso difícil para NLP)

> ⚠️ El campo `intencion_plantilla` se guarda como *ground truth* oculto para que en el ETL podamos medir la **precisión** del clasificador y mostrarlo en la presentación.

In [4]:
# Cada tupla es (texto_plantilla, intencion_original)
# El texto usa {modelo} como placeholder

PLANTILLAS_RESENAS = [
    # ===== 15 FUERTEMENTE POSITIVAS =====
    ("Me encantaron mis {modelo} 😍🔥 súper cómodos para todo el día!! #Panam #Calzado", "positivo"),
    ("BUENÍSIMOS los {modelo}!! Aguantaron toda la maratón ❤️‍🔥 100% recomendados 💯", "positivo"),
    ("Quedé enamoradísima de mis {modelo} ✨ buenísima calidad y precio 👌🥰", "positivo"),
    ("Por fin unos tenis que duran! Los {modelo} ya tienen 8 meses y siguen como nuevos 💪", "positivo"),
    ("Compré los {modelo} para mi hijo y le ENCANTARON 🥰 muy cómodos y bonitos", "positivo"),
    ("Mejor compra del año!! Los {modelo} son una BELLEZA 🔥🔥🔥 #PanamForever", "positivo"),
    ("Súper recomendados los {modelo}, no me los quería quitar 😂❤️", "positivo"),
    ("Excelentísimos los {modelo} ✨ los uso para correr y son perfectos 🏃‍♂️💨", "positivo"),
    ("Mis {modelo} son LO MEJOR que he comprado en años 😍 calidad/precio insuperable ⭐⭐⭐⭐⭐", "positivo"),
    ("AMO mis {modelo} 💖 ya pedí otro par en otro color, hermosos", "positivo"),
    ("Los mejores tenis que he tenido!! Los {modelo} son cómodos, bonitos y baratos 👌🔥", "positivo"),
    ("Yo súper feliz con mis {modelo} 🥹 mejor inversión del mes, los recomiendo muchísimo", "positivo"),
    ("10/10 los {modelo} ⭐⭐⭐⭐⭐ excelente atención y producto, llegaron rapidísimo", "positivo"),
    ("MARAVILLOSOS los {modelo} 😍 como caminar en nubes ☁️☁️ los amo", "positivo"),
    ("Increíbles!! Los {modelo} se ven y se sienten premium 🔝🔥 vale cada peso", "positivo"),

    # ===== 6 LEVEMENTE POSITIVAS =====
    ("Buenos los {modelo}, me gustaron 👍 cumplen lo prometido", "positivo"),
    ("Me gustaron los {modelo}, cómodos y bonitos para diario 🙂", "positivo"),
    ("Bonitos los {modelo} 👌 buen precio para la calidad", "positivo"),
    ("Muy cómodos los {modelo}, me los recomendó mi hermana ✌️ buena compra", "positivo"),
    ("Cumple lo prometido, los {modelo} están bien para el uso diario 👍🏼", "positivo"),
    ("Decentes los {modelo}, no son los mejores pero cumplen 🙂", "positivo"),

    # ===== 9 FUERTEMENTE NEGATIVAS =====
    ("PÉSIMOS los {modelo} 😡 se descosió la suela en 2 semanas 👎👎 NO COMPREN", "negativo"),
    ("Muy decepcionada con mis {modelo} 💔 esperaba muchísimo más por el precio que pagué 🙄", "negativo"),
    ("Los {modelo} son una PORQUERÍA, se rompieron al mes 🤬 quiero MI REEMBOLSO ya", "negativo"),
    ("Horrible calidad los {modelo} 😠 ni para regalar #malacompra #panam", "negativo"),
    ("NO LOS COMPREN!! Mis {modelo} se despegaron en una semana 😤 estafa total", "negativo"),
    ("Pésima atención y los {modelo} llegaron rotos 😠😠 jamás vuelvo a comprar nada de Panam", "negativo"),
    ("BASURA los {modelo} 🤮 me lastimaron los pies horrible 👎 nunca más", "negativo"),
    ("Terrible experiencia con los {modelo} 💔 mala calidad y peor servicio al cliente 😡", "negativo"),
    ("Decepción total los {modelo} 😞 se ven baratos y se rompen aún más rápido 👎", "negativo"),

    # ===== 5 LEVEMENTE NEGATIVAS =====
    ("Esperaba más de los {modelo}, la suela es muy delgada 😕 regulares", "negativo"),
    ("Los {modelo} no me convencieron, son muy regulares para el precio 🤷‍♀️", "negativo"),
    ("Me quedaron grandes los {modelo} y la talla era la correcta 😒 mala medición", "negativo"),
    ("Incomodísimos los {modelo}, me lastimaron los pies después de 1 hr 😣", "negativo"),
    ("Mala compra los {modelo}, no son lo que esperaba ☹️ poco durables", "negativo"),

    # ===== 5 NEUTRALES =====
    ("Los {modelo} están ok, nada del otro mundo 🤷", "neutro"),
    ("Normales los {modelo}, ni bien ni mal 🤷‍♂️", "neutro"),
    ("Compré los {modelo}, son tenis. Punto.", "neutro"),
    ("Los {modelo} cumplen, los uso para diario nomás", "neutro"),
    ("Regulares los {modelo}, esperaba un poquito más pero también un poquito menos", "neutro"),

    # ===== 5 MIXTAS (caso difícil para NLP) =====
    ("Bonitos los {modelo} pero la suela se desgasta rapidísimo 🤔 no sé qué pensar", "mixto"),
    ("Me gustaron los {modelo} aunque son un poco apretados al principio 😬 ya cedieron", "mixto"),
    ("Cómodos los {modelo} pero el color destiñe en 2 lavadas 😕 mitad y mitad", "mixto"),
    ("Excelente diseño los {modelo} pero algo caros para la calidad real 💸", "mixto"),
    ("Buena calidad los {modelo} pero la entrega tardó 3 semanas 😑 ojo con eso", "mixto"),
]

print(f"📝 Total de plantillas: {len(PLANTILLAS_RESENAS)}")

# Distribución por intención
from collections import Counter
dist = Counter(intencion for _, intencion in PLANTILLAS_RESENAS)
print(f"📊 Distribución de plantillas por intención:")
for intencion, n in dist.most_common():
    print(f"   - {intencion:10s}: {n} plantillas")

📝 Total de plantillas: 45
📊 Distribución de plantillas por intención:
   - positivo  : 21 plantillas
   - negativo  : 14 plantillas
   - neutro    : 5 plantillas
   - mixto     : 5 plantillas


## 5️⃣ Generador de Reseñas

**Pesos asimétricos por intención**: en la realidad las personas escriben más reseñas cuando están muy satisfechas o muy molestas, las opiniones tibias se publican menos. Por eso ponderamos:
- 50% positivas (orgánico de marca consolidada)
- 30% negativas (sesgo hacia quejas en redes)
- 12% neutras
- 8% mixtas

**Modelos populares vs nicho**: el `084` y sus variantes son los modelos icónicos de Panam, así que reciben más reseñas que modelos secundarios.

In [5]:
def generar_resenas(n: int = 1000) -> pd.DataFrame:
    """Genera n reseñas a partir de las plantillas con metadata realista."""
    
    # Pesos por intención (reflejan distribución real en redes sociales)
    pesos_intencion = {'positivo': 0.50, 'negativo': 0.30, 'neutro': 0.12, 'mixto': 0.08}
    
    # Pesos por modelo (los 084 son los más populares)
    pesos_modelo = []
    for m in MODELS:
        if m.startswith('084'):
            pesos_modelo.append(3.0)  # Líder de ventas
        elif m in ['Panam Alfa', 'Vaivén', 'Urbano']:
            pesos_modelo.append(2.0)  # Populares
        else:
            pesos_modelo.append(1.0)  # Resto
    pesos_modelo = np.array(pesos_modelo) / sum(pesos_modelo)
    
    # Indexar plantillas por intención para muestreo ponderado
    plantillas_por_intencion = {}
    for texto, intencion in PLANTILLAS_RESENAS:
        plantillas_por_intencion.setdefault(intencion, []).append(texto)
    
    resenas = []
    for _ in range(n):
        # 1. Elegir intención según pesos realistas
        intencion = np.random.choice(
            list(pesos_intencion.keys()),
            p=list(pesos_intencion.values())
        )
        
        # 2. Tomar una plantilla al azar de esa intención
        plantilla = random.choice(plantillas_por_intencion[intencion])
        
        # 3. Elegir modelo (ponderado por popularidad)
        modelo = np.random.choice(MODELS, p=pesos_modelo)
        
        # 4. Renderizar el comentario sustituyendo el placeholder
        comentario = plantilla.format(modelo=modelo)
        
        # 5. Metadata adicional
        seguidores = int(np.random.lognormal(mean=5, sigma=2))  # Distribución realista
        likes = int(np.random.exponential(50)) if intencion != 'neutro' else int(np.random.exponential(15))
        
        resena = {
            'usuario':              fake.user_name(),
            'usuario_seguidores':   seguidores,
            'modelo':               modelo,
            'color':                random.choice(COLORS),
            'calificacion':         _calif_desde_intencion(intencion),
            'comentario':           comentario,
            'fuente':               random.choice(FUENTES),
            'likes':                likes,
            'verificado':           random.random() < 0.65,  # 65% son compradores verificados
            'respondido_marca':     random.random() < 0.20,  # Marca contesta ~20% de las reseñas
            'fecha':                fake.date_time_between(start_date='-5y', tzinfo=None),
            'intencion_plantilla':  intencion,  # GROUND TRUTH (el ETL no debe usarlo, sólo validar)
        }
        resenas.append(resena)
    
    return pd.DataFrame(resenas)


def _calif_desde_intencion(intencion: str) -> int:
    """Mapea intención a calificación 1-5 con algo de ruido (la gente no es 100% consistente)."""
    if intencion == 'positivo':
        return random.choices([4, 5, 3], weights=[0.30, 0.65, 0.05])[0]
    if intencion == 'negativo':
        return random.choices([1, 2, 3], weights=[0.55, 0.35, 0.10])[0]
    if intencion == 'mixto':
        return random.choices([3, 2, 4], weights=[0.55, 0.25, 0.20])[0]
    return random.choices([3, 2, 4], weights=[0.70, 0.15, 0.15])[0]  # neutro

print("✅ Función generar_resenas() definida")

✅ Función generar_resenas() definida


## 6️⃣ Generador de Ventas

**Tallas mexicanas**: Panam vende del rango infantil (12-21) al adulto (22-30). Usamos una distribución que mezcla ambas:
- 30% infantil (centrada en talla 18, niños/niñas)
- 70% adulto (centrada en talla 25.5, donde está la media demográfica mexicana)

**Estacionalidad**: las ventas suben en regreso a clases (julio-agosto) y diciembre.  
**Sucursales urbanas vs regionales**: las CDMX/EdoMex venden más volumen.  
**Métodos de pago**: efectivo sigue dominando en México pero la tarjeta crece año con año.

In [6]:
# Tallas posibles (escala mexicana 12-30)
TALLAS_INFANTIL = list(range(12, 22))   # 12 a 21
TALLAS_ADULTO   = list(range(22, 31))   # 22 a 30

METODOS_PAGO = ['efectivo', 'tarjeta_debito', 'tarjeta_credito', 'transferencia', 'mercado_pago']
TIPOS_CLIENTE = ['nuevo', 'recurrente', 'vip']


def _muestrear_talla() -> int:
    """Talla con distribución realista: 30% infantil, 70% adulto."""
    if random.random() < 0.30:
        # Infantil: distribución normal centrada en 18
        t = int(np.clip(np.random.normal(18, 2), 12, 21))
    else:
        # Adulto: distribución normal centrada en 25.5 (media mexicana)
        t = int(np.clip(np.random.normal(25.5, 1.8), 22, 30))
    return t


def _peso_sucursal(sucursal: str) -> float:
    """Sucursales más urbanas venden más volumen."""
    if sucursal.startswith('CDMX'):
        return 3.0
    if sucursal.startswith('EdoMex'):
        return 2.0
    if sucursal.startswith(('Nuevo_Leon', 'Puebla', 'Quintana_Roo')):
        return 1.5
    return 1.0


def _factor_estacional(fecha: datetime) -> float:
    """Picos en regreso a clases (jul-ago) y temporada navideña (dic)."""
    mes = fecha.month
    if mes in (7, 8):     return 1.5  # Regreso a clases
    if mes == 12:         return 1.4  # Navidad
    if mes in (5, 6):     return 0.8  # Valles
    return 1.0


def generar_ventas(n: int = 5000) -> pd.DataFrame:
    """Genera n ventas distribuidas en los últimos 5 años."""
    
    pesos_suc = np.array([_peso_sucursal(s) for s in SUCURSALES])
    pesos_suc = pesos_suc / pesos_suc.sum()
    
    ventas = []
    for _ in range(n):
        sucursal = np.random.choice(SUCURSALES, p=pesos_suc)
        modelo   = random.choice(MODELS)
        talla    = _muestrear_talla()
        fecha    = fake.date_time_between(start_date='-5y', tzinfo=None)
        
        # Cantidad afectada por estacionalidad
        base_cant = max(1, int(np.random.poisson(2.5)))
        cantidad  = max(1, int(base_cant * _factor_estacional(fecha)))
        
        # Precio depende del modelo (los premium cuestan más)
        if modelo in ('084 Diamante', 'Panam Alfa', 'SuperFaro'):
            precio = random.randint(1500, 2200)
        elif modelo in ('Ultra', 'Caimán', 'Perestroika'):
            precio = random.randint(1100, 1700)
        else:
            precio = random.randint(700, 1400)
        
        # Descuento ocasional
        descuento = random.choice([0, 0, 0, 0, 0, 5, 10, 15, 20, 30])  # ~50% sin descuento
        
        ventas.append({
            'sucursal':         sucursal,
            'modelo':           modelo,
            'talla':            talla,
            'fecha':            fecha.date(),
            'hora':             fecha.time(),
            'cantidad':         cantidad,
            'precio_unitario':  precio,
            'descuento_pct':    descuento,
            'metodo_pago':      random.choices(METODOS_PAGO, weights=[0.45, 0.20, 0.15, 0.10, 0.10])[0],
            'tipo_cliente':     random.choices(TIPOS_CLIENTE, weights=[0.45, 0.45, 0.10])[0],
            'vendedor':         fake.first_name(),
        })
    
    return pd.DataFrame(ventas)

print("✅ Función generar_ventas() definida")

✅ Función generar_ventas() definida


## 7️⃣ Generador de Inventario

Como pediste, los **almacenes son los estados** de las sucursales (única fuente de verdad). Esto facilita el análisis de stock por región y el merge con ventas en el ETL.

Cada movimiento puede ser **entrada** (recepción de fábrica), **salida** (envío a sucursal) o **devolución** (cliente regresa producto).

In [7]:
TIPOS_MOVIMIENTO = ['entrada', 'salida', 'devolución']

def generar_inventario(n: int = 2000) -> pd.DataFrame:
    """Genera n registros de inventario por estado."""
    inventario = []
    for _ in range(n):
        # Tallas mismas que ventas
        talla = _muestrear_talla()
        movimiento = random.choices(
            TIPOS_MOVIMIENTO,
            weights=[0.40, 0.50, 0.10]  # más salidas que entradas
        )[0]
        
        # El stock_actual depende del tipo de movimiento
        if movimiento == 'entrada':
            stock = random.randint(100, 500)
        elif movimiento == 'salida':
            stock = random.randint(0, 300)
        else:  # devolución
            stock = random.randint(50, 250)
        
        inventario.append({
            'almacen':         random.choice(ESTADOS),  # ⭐ estados, no ciudades
            'modelo':          random.choice(MODELS),
            'talla':           talla,
            'fecha':           fake.date_time_between(start_date='-5y', tzinfo=None).date(),
            'stock':           stock,
            'capacidad_max':   random.choice([500, 750, 1000, 1500]),
            'movimiento':      movimiento,
            'punto_reorden':   random.choice([50, 75, 100, 150]),
        })
    return pd.DataFrame(inventario)

print("✅ Función generar_inventario() definida")

✅ Función generar_inventario() definida


## 8️⃣ Generador de Eventos Web

Eventos de comportamiento del usuario en el sitio web/app. Útil para:
- Detectar **carritos abandonados** y disparar campañas de remarketing
- Análizar el **funnel de conversión** (búsqueda → click → carrito → compra)
- Identificar **modelos populares** que generan tráfico aunque no se vendan tanto

In [8]:
TIPOS_EVENTO = [
    'busqueda', 'click_producto', 'agregar_carrito',
    'carrito_abandonado', 'compra_completada', 'wishlist'
]

DISPOSITIVOS = ['mobile', 'desktop', 'tablet']
NAVEGADORES = ['Chrome', 'Safari', 'Firefox', 'Edge', 'Samsung Internet']

def generar_eventos_web(n: int = 800) -> pd.DataFrame:
    """Genera n eventos web heterogéneos."""
    eventos = []
    for _ in range(n):
        eventos.append({
            'usuario_id':       f'user_{random.randint(1000, 9999)}',
            'evento_tipo':      random.choices(
                TIPOS_EVENTO,
                weights=[0.30, 0.25, 0.15, 0.15, 0.10, 0.05]
            )[0],
            'modelo':           random.choice(MODELS),
            'talla':            _muestrear_talla(),
            'duracion_sesion':  round(random.uniform(0.5, 30), 1),  # minutos
            'dispositivo':      random.choices(DISPOSITIVOS, weights=[0.65, 0.30, 0.05])[0],
            'navegador':        random.choice(NAVEGADORES),
            'estado_origen':    random.choice(ESTADOS),
            'fecha':            fake.date_time_between(start_date='-5y', tzinfo=None),
        })
    return pd.DataFrame(eventos)

print("✅ Función generar_eventos_web() definida")

✅ Función generar_eventos_web() definida


## 9️⃣ Ejecución y guardado

Ahora ejecutamos los 4 generadores y guardamos los CSVs en `data/`.

In [9]:
print("🔄 Generando datasets...\n")

df_resenas    = generar_resenas(1000)
print(f"✅ Reseñas:     {len(df_resenas):>5} registros")

df_ventas     = generar_ventas(5000)
print(f"✅ Ventas:      {len(df_ventas):>5} registros")

df_inventario = generar_inventario(2000)
print(f"✅ Inventario:  {len(df_inventario):>5} registros")

df_eventos    = generar_eventos_web(800)
print(f"✅ Eventos web: {len(df_eventos):>5} registros")

total = len(df_resenas) + len(df_ventas) + len(df_inventario) + len(df_eventos)
print(f"\n📊 TOTAL: {total:,} registros generados")

🔄 Generando datasets...

✅ Reseñas:      1000 registros
✅ Ventas:       5000 registros
✅ Inventario:   2000 registros
✅ Eventos web:   800 registros

📊 TOTAL: 8,800 registros generados


In [10]:
# Guardar CSVs
df_resenas.to_csv(   DATOS / 'resenas.csv',    index=False)
df_ventas.to_csv(    DATOS / 'ventas.csv',     index=False)
df_inventario.to_csv(DATOS / 'inventario.csv', index=False)
df_eventos.to_csv(   DATOS / 'eventos.csv',    index=False)

print(f"💾 4 archivos CSV guardados en {DATOS}/")

💾 4 archivos CSV guardados en c:\Users\PC\Desktop\Antigravity\Panam_BDN\data/


## 🔟 Vista previa de los datasets generados

In [11]:
print("=" * 80)
print("📝 RESEÑAS — muestra (notar variedad de emojis, CAPS, hashtags):")
print("=" * 80)
for _, row in df_resenas.sample(5, random_state=1).iterrows():
    print(f"\n👤 @{row['usuario']:20s} | {row['modelo']:18s} | {row['fuente']:14s} | ❤️  {row['likes']}")
    print(f"   {row['comentario']}")

📝 RESEÑAS — muestra (notar variedad de emojis, CAPS, hashtags):

👤 @cristinarolon        | 084 Campeón        | google_reviews | ❤️  21
   Me encantaron mis 084 Campeón 😍🔥 súper cómodos para todo el día!! #Panam #Calzado

👤 @marisolescobedo      | 084                | instagram      | ❤️  4
   Súper recomendados los 084, no me los quería quitar 😂❤️

👤 @socorro47            | 084                | facebook       | ❤️  79
   BASURA los 084 🤮 me lastimaron los pies horrible 👎 nunca más

👤 @anaruelas            | 084 Clásico        | amazon         | ❤️  55
   Horrible calidad los 084 Clásico 😠 ni para regalar #malacompra #panam

👤 @arredondoalejandra   | Urbano             | twitter        | ❤️  4
   AMO mis Urbano 💖 ya pedí otro par en otro color, hermosos


In [13]:
print("=" * 80)
print("💰 VENTAS — vista previa:")
print("=" * 80)
print(df_ventas.head(8).to_string(index=False))

print("\n📊 Distribución de tallas vendidas:")
print(df_ventas['talla'].value_counts().sort_index().to_string())

💰 VENTAS — vista previa:
                    sucursal       modelo  talla      fecha     hora  cantidad  precio_unitario  descuento_pct     metodo_pago tipo_cliente   vendedor
       CDMX_Rep_del_Salvador  084 Campeón     16 2026-04-07 19:39:48         1             1244              5    mercado_pago        nuevo Jorge Luis
   EdoMex_Sendero_Ixtapaluca        Ultra     24 2022-06-09 04:21:40         1             1373              0        efectivo   recurrente   Fernando
       CDMX_Metrobus_Durango  Perestroika     27 2024-08-21 22:34:29         6             1347             30 tarjeta_credito   recurrente     Úrsula
           CDMX_Plaza_Ermita       Vaivén     25 2024-01-31 05:55:54         3              710              0   transferencia        nuevo      Juana
            CDMX_Insurgentes 084 Diamante     29 2024-08-13 10:02:48         1             1956              5        efectivo        nuevo      Diana
        Nuevo_Leon_Monterrey       Urbano     15 2023-06-16 13:32:50 

In [14]:
print("=" * 80)
print("📦 INVENTARIO — vista previa (almacenes = estados):")
print("=" * 80)
print(df_inventario.head(8).to_string(index=False))

print(f"\n🗺️  Almacenes distintos: {sorted(df_inventario['almacen'].unique())}")

📦 INVENTARIO — vista previa (almacenes = estados):
  almacen       modelo  talla      fecha  stock  capacidad_max movimiento  punto_reorden
 Coahuila       Urbano     22 2025-09-26    215           1500     salida            100
Chihuahua    SuperFaro     16 2025-05-17    185           1500     salida             75
Chihuahua          084     23 2024-11-21     24            750     salida            150
 Quintana         Flip     28 2023-08-21     29            750     salida             50
 Coahuila 084 Diamante     15 2023-04-16    166            750 devolución             50
 Quintana  084 Campeón     26 2023-07-31     38           1000     salida            100
  Hidalgo       Caimán     28 2025-05-11    256            750    entrada             75
  Morelos       Urbano     20 2023-12-05    317            750    entrada             75

🗺️  Almacenes distintos: ['CDMX', 'Chihuahua', 'Coahuila', 'EdoMex', 'Hidalgo', 'Michoacan', 'Morelos', 'Nuevo', 'Puebla', 'Quintana']


In [15]:
print("=" * 80)
print("🌐 EVENTOS WEB — vista previa:")
print("=" * 80)
print(df_eventos.head(8).to_string(index=False))

print("\n📊 Tipos de evento más frecuentes:")
print(df_eventos['evento_tipo'].value_counts().to_string())

🌐 EVENTOS WEB — vista previa:
usuario_id        evento_tipo      modelo  talla  duracion_sesion dispositivo        navegador estado_origen               fecha
 user_1443    agregar_carrito 084 Clásico     25             16.0      mobile Samsung Internet      Quintana 2021-06-25 16:58:20
 user_5203     click_producto       Ultra     26             13.5      mobile             Edge      Coahuila 2021-08-10 22:18:53
 user_1902           busqueda       Ultra     23              9.5      mobile           Chrome         Nuevo 2023-11-11 06:46:50
 user_6910     click_producto       Ultra     24             28.3     desktop          Firefox     Chihuahua 2022-05-17 11:34:04
 user_5644     click_producto   Barracuda     27             24.1      mobile Samsung Internet     Chihuahua 2023-02-08 17:04:37
 user_5790 carrito_abandonado      Vaivén     26             21.2      mobile Samsung Internet        Puebla 2022-12-04 07:22:10
 user_7453           busqueda        Flip     26             26.6  

## ✅ Resumen

Datos generados con realismo apto para minería de texto y análisis temporal:

- **45 plantillas de reseñas** con emojis, CAPS, hashtags y casos mixtos
- **49 sucursales reales**, **10 estados**, **16 modelos**
- **Tallas mexicanas 12-30** con distribución mezcla infantil/adulto
- **Fechas de los últimos 5 años** para análisis estacionales
- **Campos extra** para enriquecer el dashboard: seguidores, descuentos, dispositivos, métodos de pago, etc.

➡️ Siguiente paso: ejecutar `02_etl_transformaciones.ipynb` para que el sentimiento se **derive del texto** vía análisis léxico + scoring de emojis.